In [48]:
"""SPLADE sparse embedding client using Triton Inference Server."""

from __future__ import annotations

from typing import Any, cast
import pickle
import numpy as np
from loguru import logger
from pydantic import BaseModel
from qdrant_client.models import SparseVector
from tritonclient.grpc import InferenceServerClient, InferInput, InferRequestedOutput, InferResult 


class SpladeConfig(BaseModel):
    """Configuration for SPLADE Triton client."""

    url: str = "localhost:8001"
    model_name: str = "splade"
    timeout: float = 30
    verbose: bool = False


class SpladeClient:
    def __init__(self, config: SpladeConfig):
        self.config = config
        self._client: InferenceServerClient | None = None

    def _get_client(self) -> InferenceServerClient:
        if self._client is None:
            self._client = InferenceServerClient(
                url=self.config.url,
                verbose=self.config.verbose,
            )
        return self._client

    def close(self) -> None:
        if self._client is not None:
            self._client.close()
            self._client = None

    def __enter__(self) -> "SpladeClient":
        return self

    def __exit__(self, exc_type: Any, exc_val: Any, exc_tb: Any) -> None:
        self.close()

    def check_health(self) -> bool:
        try:
            client = self._get_client()

            if not client.is_server_live():
                logger.error("[SpladeClient] Triton server is not live")
                return False

            if not client.is_model_ready(self.config.model_name):
                logger.error(
                    f"[SpladeClient] Model '{self.config.model_name}' is not ready"
                )
                return False

            logger.success(
                f"[SpladeClient] Triton server healthy, model '{self.config.model_name}' ready"
            )
            return True

        except Exception as e:
            logger.error(f"[SpladeClient] Health check failed: {e}")
            return False

    def encode(self, texts: list[str]) -> list[SparseVector]:
        if not texts:
            return []

        client = self._get_client()

    
        text_input = InferInput("TEXT", [len(texts), 1], "BYTES")
        text_input.set_data_from_numpy(np.array([[t] for t in texts], dtype=object))

        outputs = [
            InferRequestedOutput("INDICES"),
            InferRequestedOutput("VALUES"),
        ]

        try:
            result = client.infer(
                model_name=self.config.model_name,
                inputs=[text_input],
                outputs=outputs,
                timeout=self.config.timeout,
            )
        except Exception as e:
            logger.error(f"[SpladeClient] Inference failed: {e}")
            raise RuntimeError(f"SPLADE inference failed: {e}") from e

        if result is None:
            logger.error(f"[SpladeClient] Inference failed: result return is None for some reason")
            raise RuntimeError(f"SPLADE inference failed. Result return None")
        
        
        indices_batch = result.as_numpy("INDICES")
        values_batch = result.as_numpy("VALUES")
        
        print(indices_batch)
        
        assert indices_batch is not None, "[SpladeClient] Indices batch is None for some reason"
        assert values_batch is not None, "[SpladeClient] Value batch is None for some reason"

        sparse_vectors = []
        for i in range(len(texts)):
            indices = pickle.loads(indices_batch[i]).tolist()
            values = pickle.loads(values_batch[i]).tolist()

            sparse_vectors.append(
                SparseVector(
                    indices=indices,
                    values=values,
                )
            )

        logger.debug(
            f"[SpladeClient] Encoded {len(texts)} text(s) to sparse vectors"
        )
        return sparse_vectors

    async def aencode(self, texts: list[str]) -> list[SparseVector]:
        return self.encode(texts)



In [49]:
texts = [
    "This is a test.",
    "Hello, world!",
]
client = SpladeClient(SpladeConfig())

In [50]:
client.encode(texts)

2026-03-16 13:13:03.384 | DEBUG    | __main__:encode:121 - [SpladeClient] Encoded 2 text(s) to sparse vectors


[b'\x80\x04\x95\xed\x00\x00\x00\x00\x00\x00\x00\x8c\x16numpy._core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01K\x19\x85\x94h\x03\x8c\x05dtype\x94\x93\x94\x8c\x02i4\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89Cd\r\x04\x00\x00\xd3\x07\x00\x00\xe7\x07\x00\x00J\x08\x00\x00o\n\x00\x00\x10\x0b\x00\x00\x9f\x0c\x00\x00\xc3\x0c\x00\x00G\r\x00\x00\xd8\x0e\x00\x00\n\x10\x00\x00\x1e\x12\x00\x00\xd8\x13\x00\x00\xe4\x15\x00\x00\xdc\x16\x00\x00B\x18\x00\x00\xa9\x1a\x00\x00\xbd\x1a\x00\x00\x7f\x1d\x00\x00\xf3\x1d\x00\x00\x1d\x1e\x00\x00&\x1e\x00\x00J!\x00\x00`$\x00\x00`,\x00\x00\x94t\x94b.'
 b'\x80\x04\x95\xd9\x00\x00\x00\x00\x00\x00\x00\x8c\x16numpy._core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01K\x14\x85\x94h\x03\x8c\x05dtype\x94\x93\x94\x8c\x02i4\x94\x89\x88\x87\x94R\x94(

[SparseVector(indices=[1037, 2003, 2023, 2122, 2671, 2832, 3231, 3267, 3399, 3800, 4106, 4638, 5080, 5604, 5852, 6210, 6825, 6845, 7551, 7667, 7709, 7718, 8522, 9312, 11360], values=[1.2529109716415405, 0.5843831300735474, 1.7758288383483887, 0.5920519828796387, 0.15831078588962555, 0.15472523868083954, 2.9668307304382324, 1.279798150062561, 0.14651966094970703, 0.3536541759967804, 0.33246704936027527, 0.5830166935920715, 0.019948923960328102, 1.7542455196380615, 1.6737236976623535, 0.7712801098823547, 0.02305162325501442, 2.010756731033325, 0.07071647047996521, 0.0060702720656991005, 0.7976645231246948, 0.18701238930225372, 0.3480924665927887, 0.2453208863735199, 0.9125958681106567]),
 SparseVector(indices=[999, 2017, 2026, 2040, 2057, 2088, 2272, 2299, 2477, 2748, 3376, 3795, 4774, 5304, 6160, 6919, 7592, 7632, 8404, 8484], values=[1.9145121574401855, 0.0618186891078949, 0.09442538768053055, 0.35482174158096313, 0.0037486536893993616, 2.8006772994995117, 0.06492841988801956, 0.406799